In [ ]:
import argparse
import pandas as pd
import torch
import os
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, PreTrainedTokenizerFast

from models import Model1_GloVe, Model2_GloVeAttn, Model3_FrozenBERT, Model4_TunedBERT
from dataset import NMTDataset, nmt_collate_fn, normalize_text
from decode import beam_search_decode

In [ ]:
def main():
    parser = argparse.ArgumentParser(description="English-Hindi Translation Inference Script")
    parser.add_argument("--input", type=str, required=True, help="Path to input CSV")
    parser.add_argument("--output", type=str, required=True, help="Path to output CSV")
    parser.add_argument("--checkpoint", type=str, required=True, help="Directory containing checkpoints")
    parser.add_argument("--mode", type=int, required=True, choices=[1, 2, 3, 4], help="Model architecture ID")
    parser.add_argument("--decoding_strategy", type=str, required=True, choices=['beam', 'greedy'], help="Decoding strategy")
    
    args = parser.parse_args()
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    tgt_tokenizer_path = os.path.join(args.checkpoint, "tokenizer_target")
    if not os.path.exists(tgt_tokenizer_path):
        tgt_tokenizer_path = args.checkpoint
        
    tgt_tokenizer = PreTrainedTokenizerFast.from_pretrained(tgt_tokenizer_path)
    
    if args.mode in [1, 2]:
        glove_ckpt_path = os.path.join(args.checkpoint, "glove_vocab.pt")
        if not os.path.exists(glove_ckpt_path):
            raise FileNotFoundError(f"glove_vocab.pt not found in {args.checkpoint}")
            
        glove_ckpt = torch.load(glove_ckpt_path, map_location='cpu')
        glove_matrix = glove_ckpt["matrix"]
        glove_word2idx = glove_ckpt["word2idx"]
        
        if args.mode == 1:
            model = Model1_GloVe(glove_matrix, tgt_tokenizer.vocab_size)
        else:
            model = Model2_GloVeAttn(glove_matrix, tgt_tokenizer.vocab_size)
            
        src_mode = "glove"
        bert_tokenizer = None
    else:
        bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")
        if args.mode == 3:
            model = Model3_FrozenBERT(tgt_tokenizer.vocab_size)
        else:
            model = Model4_TunedBERT(tgt_tokenizer.vocab_size)
            
        src_mode = "bert"
        glove_word2idx = None

    model_path = os.path.join(args.checkpoint, f"model_{args.mode}.pth")
    if not os.path.exists(model_path):
        possible = [f"best_Model{args.mode}.pth", f"best_Model{args.mode}_GloVe.pth", 
                    f"best_Model{args.mode}_GloVeAttn.pth", f"best_Model{args.mode}_FrozenBERT.pth", 
                    f"best_Model{args.mode}_TunedBERT.pth"]
        found = False
        for p_name in possible:
            p = os.path.join(args.checkpoint, p_name)
            if os.path.exists(p):
                model_path = p
                found = True
                break
        if not found:
            raise FileNotFoundError(f"Weights for model mode {args.mode} not found in {args.checkpoint}")

    print(f"Loading weights from {model_path}...")
    state_dict = torch.load(model_path, map_location=device)
    model.load_state_dict(state_dict)
    model.to(device)
    model.eval()

    df = pd.read_csv(args.input)
    if "source" not in df.columns:
        raise ValueError("Input CSV must contain a 'source' column")
    
    temp_df = df.copy()
    if "translated" not in temp_df.columns:
        temp_df["translated"] = ""
    
    temp_input_path = "temp_inference_input.csv"
    temp_df.to_csv(temp_input_path, index=False)
    
    dataset = NMTDataset(
        temp_input_path, tgt_tokenizer, 
        src_tokenizer_mode=src_mode, 
        glove_word2idx=glove_word2idx, 
        bert_tokenizer=bert_tokenizer
    )
    
    dataloader = DataLoader(dataset, batch_size=1, shuffle=False, collate_fn=nmt_collate_fn)
    
    beam_size = 5 if args.decoding_strategy == 'beam' else 1
    
    print(f"Starting inference with strategy={args.decoding_strategy} (beam_size={beam_size})...")
    translations = []
    with torch.no_grad():
        for batch in dataloader:
            src_ids = batch["src_ids"].to(device)
            src_mask = batch["src_attention_mask"].to(device)
            
            # decode_dataset logic but batch-wise for 1 item
            best_seq = beam_search_decode(
                model, src_ids, src_mask, 
                beam_size=beam_size, 
                max_tgt_len=128
            )
            pred_text = tgt_tokenizer.decode(best_seq, skip_special_tokens=True)
            translations.append(pred_text)
                
    if os.path.exists(temp_input_path):
        os.remove(temp_input_path)
        
    df["translated"] = translations
    df.to_csv(args.output, index=False)
    print(f"Successfully saved {len(translations)} translations to {args.output}")

In [ ]:
if __name__ == "__main__":
    main()